In [1]:
import requests
import pandas as pd
import time

In [2]:
# Get the data from the API
url = 'https://api.legiscan.com/?key='
key = 'b88b91590f17092f1dc4e4b6147776d0' # insert your key here
page = 1
request = requests.get(url + key + '&op=getSearch&state=ALL&query=transgender' + '&page=' + str(page))


In [3]:
# get the page_total and the count from the request summary
page_total = request.json()['searchresult']['summary']['page_total']
count = request.json()['searchresult']['summary']['count']
print('Page total: ' + str(page_total) + '\n' + 'Total results: ' + str(count))

Page total: 14
Total results: 661


In [22]:
# request the additional pages of the query by adding 1 to the 'page' 
# parameter until it reaches the page_total. Store each page of  
# requests in a list. Wait 3 seconds between each request to avoid 
# overloading the API

pages = []
for i in range(page_total):
    page = i + 1
    request = requests.get(url + key + '&op=getSearch&state=ALL&query=transgender' + '&page=' + str(page))
    time.sleep(3)
    pages.append(request.json())


In [5]:
# for each page of the request, parse the results and add them to a 
# dataframe. each page is a json file with individual results labeled
# '0' through '49' and # 'summary' nested under searchresult. Ignore 
# the summary and use pandas.concat to add each of the results from 
# the request in a dataframe

df = pd.DataFrame()
for page in pages:
    results = page['searchresult']
    # if the page has no results, skip it
    for i in range(50):
        if str(i) in results:
            df = pd.concat([df, pd.DataFrame(results[str(i)], index=[i])])
        else:
            continue


In [23]:
# create a list of bill_ids from the dataframe

bill_ids = df['bill_id'].tolist()
bill_ids[:5]

# get the bill details from the bill_id: 
# https://api.legiscan.com/?key=APIKEY&op=getBill&id=BILL_ID

bill_details = []
for bill_id in bill_ids:
    request = requests.get(url + key + '&op=getBill&id=' + str(bill_id))
    bill_details.append(request.json())


KeyboardInterrupt: 

In [8]:
# Create a DataFrame adding each bill from bill_details 

bill_df = pd.DataFrame()
for i in range(len(bill_details)):
    bill_df = pd.concat([bill_df, pd.json_normalize(bill_details[i]['bill'], record_path=None)])

In [11]:
bill_df['texts']

0    [{'doc_id': 2841493, 'date': '2023-09-18', 'ty...
0    [{'doc_id': 2619414, 'date': '2022-12-19', 'ty...
0    [{'doc_id': 2769857, 'date': '2023-04-03', 'ty...
0    [{'doc_id': 2837562, 'date': '2023-08-16', 'ty...
0    [{'doc_id': 2832366, 'date': '2023-07-01', 'ty...
                           ...                        
0    [{'doc_id': 2671653, 'date': '2023-01-31', 'ty...
0    [{'doc_id': 2784910, 'date': '2023-04-19', 'ty...
0    [{'doc_id': 2675728, 'date': '2023-02-01', 'ty...
0    [{'doc_id': 2729845, 'date': '2023-02-01', 'ty...
0    [{'doc_id': 2729461, 'date': '2023-02-01', 'ty...
Name: texts, Length: 640, dtype: object

In [12]:
# get the first element of the 'texts' dict in the tenth row of bill_df
bill_df['texts'].iloc[10][-1]


{'doc_id': 2649219,
 'date': '2023-01-18',
 'type': 'Introduced',
 'type_id': 1,
 'mime': 'text/html',
 'mime_id': 1,
 'url': 'https://legiscan.com/NY/text/A01708/id/2649219',
 'state_link': 'https://assembly.state.ny.us/leg/?default_fld=&bn=A01708&term=2023&Summary=Y&Actions=Y&Text=Y&Committee%26nbspVotes=Y&Floor%26nbspVotes=Y#A01708',
 'text_size': 11095,
 'text_hash': '7bd73c24efbfc05e782719d28e174b9e',
 'alt_bill_text': 0,
 'alt_mime': '',
 'alt_mime_id': 0,
 'alt_state_link': '',
 'alt_text_size': 0,
 'alt_text_hash': ''}

In [15]:
# for each bill in bill_df, add a column, latest_text_id, 
# with the 'doc_id' from the last element of the texts list. 

# Also add a column, latest_text_type, with the 'type' from 
# the last element of the texts list if the columns don't 
# already exist, create them

if 'latest_text_id' not in bill_df.columns:
    bill_df['latest_text_id'] = ''
if 'latest_text_type' not in bill_df.columns:
    bill_df['latest_text_type'] = ''
for i in range(len(bill_df)):
    bill_df['latest_text_id'].iloc[i] = bill_df['texts'].iloc[i][-1]['doc_id']
    bill_df['latest_text_type'].iloc[i] = bill_df['texts'].iloc[i][-1]['type']
        
    

/Users/filipacalado/opt/anaconda3/lib/python3.7/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


In [20]:
# do a left join of the df and bill_df dataframes on the bill_id column.
# keep all columns from df and only the 'status', 'status_date', 'body',
# "description", 'session.session_name', 'latest_text_id' and 
# 'latest_text_type' columns from bill_df

joined_df = df.merge(bill_df[[
    'bill_id', 
    'status', 
    'status_date', 
    'body', 
    "description", 
    'session.session_name', 
    'latest_text_id', 
    'latest_text_type'
]], on='bill_id', how='left')

In [17]:
# save the joined_df dataframe to csv files
joined_df.to_csv('trans_bills.csv')

In [18]:
# reset the index for saving 
bill_df = bill_df.reset_index(drop=True)
# save the bill_df dataframe to a json file
bill_df.to_json('trans_bills_full_details.json', orient='records')


In [21]:
# get the text of the latest version of each bill in joined_df from the API following this format: https://api.legiscan.com/?key=APIKEY&op=getBillText&id=DOC_ID where DOC_ID == latest_text_id
# create a list of doc_id's from the dataframe

doc_ids = joined_df['latest_text_id'].tolist()

# get the bill text 
bill_texts = []
for doc_id in doc_ids:
    request = requests.get(url + key + '&op=getBillText&id=' + str(doc_id))
    bill_texts.append(request.json())


In [24]:
# create a dataframe from the bill_texts json file
bill_texts_df = pd.DataFrame(bill_texts)

In [ ]:
# save the bill_texts to a json file
bill_texts_df.to_json('trans_bill_texts.json', orient='records')

## decoding

In [10]:
import base64
import pandas as pd

In [2]:
a = 'eW91ciB0ZXh0'

In [8]:
base64.b64decode(a)

b'your text'

In [35]:
data = pd.read_json('trans_bill_texts.json')

In [13]:
df = pd.DataFrame(data)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   status  640 non-null    object
 1   text    640 non-null    object
dtypes: object(2)
memory usage: 10.1+ KB


In [27]:
docs = df['text'].str['doc'].tolist()

In [30]:
test = docs[:5]

In [31]:
decoded = []
for i in test:
    d = base64.b64decode(i)
    decoded.append(d)

In [33]:
decoded[0]

b'%PDF-1.7\r%\xe2\xe3\xcf\xd3\r\n391 0 obj\r<</Linearized 1/L 429837/O 393/E 157012/N 20/T 429337/H [ 525 348]>>\rendobj\r            \r\n411 0 obj\r<</DecodeParms<</Columns 5/Predictor 12>>/Filter/FlateDecode/ID[<F090A3C51FF5F34293F1AE764EA1A6C4><BE4EE93B9BB06A4DA40CA3B02A6ECB35>]/Index[391 52]/Info 390 0 R/Length 110/Prev 429338/Root 392 0 R/Size 443/Type/XRef/W[1 3 1]>>stream\r\nh\xdebbd`\x10``b``\x8e\x04\x91\x0c\r \x92I\x18D\xb2\xb1\x83H\x16i\x10\xc9x\x1cD\xb2\x1e\x02\x8b\xb0\x80\xc9\xdb`\x91G`\xf20X\xc4\x00L\xea\x81H\x9f\xc5@\x92\xf1\xb9?\x88-\xa7\x0fb_]\x066\xa7\tl\xd7z \xf9o\xf9g\x06&F\x06F1\xb0\xbd\x0c\x8cC\x80\xfc\xcf\xc0\xd8\xfc\x04H\xfe\xfb\xcc\x00\x10`\x00\x81\xa6\x13i\r\nendstream\rendobj\rstartxref\r\n0\r\n%%EOF\r\n             \r\n441 0 obj\r<</C 318/Filter/FlateDecode/I 340/Length 253/O 302/S 231>>stream\r\nh\xdeb```\x02\xa2^\x06V \xb9\x8bA\x88\x01\x01\x84\x18\x98\x81\xa2,\x0c\x1c\rRH\xa2\xd3\xd78\xeaM\xdf!\xc0=\x93\xe9SC\xbf\x82\xd2\x0f\xce(&\x9f\x03\x0b\x17\x18\xe8p|e